# Notebook 01 — Set up your laptop for Kafka

**Goal of this notebook:** by the end, a Kafka server is running on your laptop and a Python cell has talked to it.

**We will NOT send any messages yet.** That is Notebook 03. Here we only build the workshop.

| Step | What you do | Time |
|---|---|---|
| 1 | Understand the 3 pieces on your laptop | 5 min |
| 2 | Check the tools you already have | 5 min |
| 3 | Install the Python Kafka library | 5 min |
| 4 | Write the file that describes the Kafka server | 10 min |
| 5 | Start Kafka | 5 min |
| 6 | Prove it is alive from Python | 5 min |
| 7 | Stop / restart / wipe | 5 min |

## Step 1 — The 3 pieces

```text
 YOUR LAPTOP
+-----------------------------------------------------------------+
|                                                                 |
|  [A] This notebook                        [B] Docker            |
|      (Python code)                            +---------------+ |
|            |                                  | [C] Kafka     | |
|            |   talks to  localhost:9092  ---> |     server    | |
|            |                                  +---------------+ |
+-----------------------------------------------------------------+
```

| Word | Plain meaning |
|---|---|
| **Kafka** | A program that stores and delivers streams of messages. It is a *server*: it sits and waits for other programs to connect. |
| **Broker** | One running Kafka server is called a *broker*. We will run exactly one. |
| **Docker** | A tool that runs programs in sealed boxes, so we do not install Kafka and Java by hand. |
| **Image** | The recipe/package of a program (here: Kafka). Downloaded once. |
| **Container** | A running copy of an image. Our container will be named `kafka-lab`. |
| **Docker Compose** | Lets us describe a container in a small text file (`docker-compose.yml`) and start it with one command. |
| **Port 9092** | A numbered "door" on your laptop. Our Python code knocks on `localhost:9092`; Docker forwards the knock to Kafka inside the box. |
| **Python client library** | Python code that knows how to speak Kafka's network language. We use `confluent-kafka`. |

## Step 2 — Check the tools you already have

We need four things: **Python**, **uv** (installs Python packages), **Docker**, **Docker Compose**.

The next cell only *looks*; it changes nothing. A `MISSING` line means: install that tool first.

In [1]:
import subprocess
import sys
from pathlib import Path


def run_on_host(command):
    """Run a command on your real machine and print its output.

    VS Code installed as a Flatpak runs this notebook inside a sandbox that cannot see
    docker or uv. 'flatpak-spawn --host' escapes the sandbox. On a normal install the
    command runs as-is. Returns the finished process so callers can check returncode.
    """
    in_flatpak_sandbox = Path("/.flatpak-info").exists()
    full_command = ["flatpak-spawn", "--host", *command] if in_flatpak_sandbox else command
    try:
        result = subprocess.run(full_command, capture_output=True, text=True)
    except FileNotFoundError:
        print(f"MISSING: program '{command[0]}' not found")
        return None
    print((result.stdout + result.stderr).strip())
    return result


print("Python:", sys.version.split()[0], "(needs 3.12 or newer)")
print("Running inside Flatpak sandbox:", Path("/.flatpak-info").exists())
print()
for label, command in [
    ("uv", ["uv", "--version"]),
    ("Docker", ["docker", "--version"]),
    ("Docker Compose", ["docker", "compose", "version"]),
]:
    print(f"--- {label}")
    run_on_host(command)

Python: 3.13.15 (needs 3.12 or newer)
Running inside Flatpak sandbox: True

--- uv
uv 0.12.13 (x86_64-unknown-linux-gnu)
--- Docker
Docker version 29.9.1, build dfc520d
--- Docker Compose
Docker Compose version v5.6.0


### Can Docker actually run? (the most common first-day problem)

Having Docker *installed* is not enough. Your user must be *allowed* to talk to it. Run the next cell.

In [2]:
result = run_on_host(["docker", "ps"])
print()
print("OK - Docker is reachable." if result and result.returncode == 0 else "PROBLEM - see message above.")

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

OK - Docker is reachable.


**If you see `permission denied ... docker.sock`** (this is what happens on a fresh Linux machine):

```bash
sudo usermod -aG docker $USER     # allow your user to use Docker
# then LOG OUT and LOG IN again (or reboot) - group changes only apply to new logins
```

**If you see `Cannot connect to the Docker daemon`**: Docker is installed but not running. Start it: `sudo systemctl start docker`.

Re-run the cell above until it prints `OK`.

## Step 3 — Install the Python Kafka library

Run the next cell (it uses `run_on_host` from Step 2). It does the same as these terminal commands, run from the project root `kafka_version_3/`:

```bash
uv add --upgrade confluent-kafka     # newest confluent-kafka, recorded in pyproject.toml
uv add --dev ipykernel               # lets this notebook run inside the project's environment
uv sync                              # make the project's .venv match
```

Then in VS Code: top-right of the notebook → **Select Kernel** → **Python Environments** → choose `.venv`.

Now verify. This cell prints which versions you really got.

In [3]:
# --project points uv at the project root (the folder holding pyproject.toml),
# so it works no matter which folder this notebook runs from.
project_root = str(Path("..").resolve())

run_on_host(["uv", "add", "--project", project_root, "--upgrade", "confluent-kafka"])
run_on_host(["uv", "add", "--project", project_root, "--dev", "ipykernel"])
# --managed-python: build .venv on a Python that uv downloads into your home folder.
# Needed because a Flatpak VS Code and your host have DIFFERENT system Pythons, and a
# .venv built on the host's one cannot be used by the notebook kernel.
run_on_host(["uv", "sync", "--project", project_root, "--python", "3.13", "--managed-python"])

Resolved 38 packages in 210ms
Checked 5 packages in 0.26ms
Resolved 38 packages in 1ms
Checked 33 packages in 0.70ms
Resolved 38 packages in 1ms
Checked 33 packages in 0.84ms


CompletedProcess(args=['flatpak-spawn', '--host', 'uv', 'sync', '--project', '/home/shubham/Projects/kafka_version_3', '--python', '3.13', '--managed-python'], returncode=0, stdout='', stderr='Resolved 38 packages in 1ms\nChecked 33 packages in 0.84ms\n')

In [4]:
import confluent_kafka

print("confluent-kafka (Python library):", confluent_kafka.version()[0])
print("librdkafka (C engine inside it) :", confluent_kafka.libversion()[0])

confluent-kafka (Python library): 2
librdkafka (C engine inside it) : 2.16.0


> `confluent-kafka` is a thin Python wrapper over **librdkafka**, a fast C library that does the real network work. That is why it prints two versions. If this cell fails with `ModuleNotFoundError`, the notebook is not using `.venv` — re-pick the kernel.

## Step 4 — Describe the Kafka server in a file

The next cell writes `notebook/lab/docker-compose.yml`. **Read the comments**; each line is explained again in the table below.

We use `apache/kafka:latest` - the official image from the Apache Kafka project, always the newest release.

In [5]:
from pathlib import Path

lab_folder = Path("lab")
lab_folder.mkdir(exist_ok=True)

compose_text = """\
services:
  kafka:
    image: apache/kafka:latest
    container_name: kafka-lab
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://:9092,CONTROLLER://:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@localhost:9093
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      KAFKA_NUM_PARTITIONS: 3
      KAFKA_LOG_DIRS: /var/lib/kafka/data
    volumes:
      - kafka_lab_data:/var/lib/kafka/data

volumes:
  kafka_lab_data:
"""

(lab_folder / "docker-compose.yml").write_text(compose_text)
print("Wrote", lab_folder / "docker-compose.yml")

Wrote lab/docker-compose.yml


### Every line, explained

**Docker lines**

| Line | Meaning |
|---|---|
| `image: apache/kafka:latest` | Which package to download. `latest` = newest release. *Trade-off:* in a real company you pin an exact version (e.g. `4.2.0`) so a surprise upgrade cannot break production. For learning, newest is what you asked for. |
| `container_name: kafka-lab` | Friendly name, so commands say `kafka-lab` instead of a random ID. |
| `ports: "9092:9092"` | `laptop-port : container-port`. Opens door 9092 on your laptop and forwards it into the box. |
| `volumes: kafka_lab_data:/var/lib/kafka/data` | Kafka stores messages as files in that folder inside the box. A *volume* keeps those files **outside** the box, so messages survive a restart. |

**Kafka settings** (the `KAFKA_...` lines become Kafka's own configuration)

| Setting | Meaning | Why this value |
|---|---|---|
| `KAFKA_NODE_ID: 1` | A unique number for this server in the cluster. | We have one server, so `1`. |
| `KAFKA_PROCESS_ROLES: broker,controller` | A Kafka server can have two jobs. **Broker** = stores and serves messages. **Controller** = keeps the cluster's records (which topics exist, who is in charge of what). | One server does both. Big clusters split the jobs. |
| `KAFKA_LISTENERS` | The doors Kafka opens *inside the box*, as `NAME://address:port`. `PLAINTEXT` door 9092 is for clients (your Python). `CONTROLLER` door 9093 is for controller-only traffic. | Empty address before `:` means "all network interfaces". |
| `KAFKA_ADVERTISED_LISTENERS` | The address Kafka **tells clients to use** when they ask "where do I connect?" | See the warning below. This is the #1 cause of "I cannot connect". |
| `KAFKA_LISTENER_SECURITY_PROTOCOL_MAP` | What security each door uses. `PLAINTEXT` = no encryption, no login. | Fine on your own laptop. Never in production. |
| `KAFKA_CONTROLLER_LISTENER_NAMES` | Which door is the controller door. | `CONTROLLER` (defined above). |
| `KAFKA_CONTROLLER_QUORUM_VOTERS: 1@localhost:9093` | The list of controllers allowed to vote: `nodeId@address:port`. | Just us. |
| `KAFKA_INTER_BROKER_LISTENER_NAME` | Door brokers use to talk to each other. | Required setting even with one broker. |
| `KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1` | Kafka keeps an internal topic of consumer progress. *Replication factor* = number of copies. | Default is 3 copies, impossible with one server, so Kafka would hang. We set `1`. |
| `KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1` and `..._MIN_ISR: 1` | Same idea for the internal transactions topic. `MIN_ISR` = minimum copies that must be in sync. | Same reason. |
| `KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0` | How long a new consumer group waits for more members before starting. | `0` makes demos instant. Default is 3000 ms. |
| `KAFKA_NUM_PARTITIONS: 3` | Default number of partitions for a new topic. | 3 lets us see partitions in action later. |
| `KAFKA_LOG_DIRS: /var/lib/kafka/data` | The folder where Kafka writes message files. | Must match the volume mount below. Without it the image writes to `/tmp/kafka-logs`, **outside** the volume, and messages vanish when the container is removed. |

> **⚠ Advertised listeners — read twice.** Connecting to Kafka is a *two-step* conversation:
> 1. Your code connects to the *bootstrap address* (`localhost:9092`) and asks, "who is in the cluster?"
> 2. Kafka answers with the **advertised** address. Your code then reconnects to *that*.
>
> If the advertised address is something your laptop cannot reach (like a Docker-internal name), step 1 succeeds and step 2 fails. Here we advertise `localhost:9092`, which is correct because your notebook runs on the laptop, not inside Docker.

## Step 5 — Start Kafka

First run downloads the image (a few hundred MB), so it takes a minute. `-d` means "detached": run in the background and give me my terminal back.

In [7]:
run_on_host(['docker', 'compose', '-f', str(Path('lab/docker-compose.yml').resolve()), 'up', '-d'])

Network lab_default Creating 
 Network lab_default Creating 
 Volume lab_kafka_lab_data Creating 
 Volume lab_kafka_lab_data Creating 
 Volume lab_kafka_lab_data Created 
 Volume lab_kafka_lab_data Created 
 Network lab_default Created 
 Network lab_default Created 
 Container kafka-lab Creating 
 Container kafka-lab Created 
 Container kafka-lab Starting 
 Container kafka-lab Started


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'compose', '-f', '/home/shubham/Projects/kafka_version_3/notebook/lab/docker-compose.yml', 'up', '-d'], returncode=0, stdout='', stderr=' Network lab_default Creating \n Network lab_default Creating \n Volume lab_kafka_lab_data Creating \n Volume lab_kafka_lab_data Creating \n Volume lab_kafka_lab_data Created \n Volume lab_kafka_lab_data Created \n Network lab_default Created \n Network lab_default Created \n Container kafka-lab Creating \n Container kafka-lab Created \n Container kafka-lab Starting \n Container kafka-lab Started \n')

Expected last lines: `Container kafka-lab  Started`.

Check that it is running and read Kafka's own startup log:

In [8]:
run_on_host(['docker', 'ps', '--filter', 'name=kafka-lab'])

CONTAINER ID   IMAGE                 COMMAND                  CREATED         STATUS         PORTS                                         NAMES
4aab4866297c   apache/kafka:latest   "/__cacert_entrypoin…"   9 seconds ago   Up 4 seconds   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp   kafka-lab


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'ps', '--filter', 'name=kafka-lab'], returncode=0, stdout='CONTAINER ID   IMAGE                 COMMAND                  CREATED         STATUS         PORTS                                         NAMES\n4aab4866297c   apache/kafka:latest   "/__cacert_entrypoin…"   9 seconds ago   Up 4 seconds   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp   kafka-lab\n', stderr='')

In [9]:
run_on_host(['docker', 'logs', '--tail', '5', 'kafka-lab'])

[2026-10-15 06:47:13,718] INFO [BrokerServer id=1] Transition from STARTING to STARTED (kafka.server.BrokerServer)
[2026-10-15 06:47:13,719] INFO Kafka version: 4.3.1 (org.apache.kafka.common.utils.AppInfoParser)
[2026-10-15 06:47:13,719] INFO Kafka commitId: 26b251a451ce941d (org.apache.kafka.common.utils.AppInfoParser)
[2026-10-15 06:47:13,719] INFO Kafka startTimeMs: 1792046833719 (org.apache.kafka.common.utils.AppInfoParser)
[2026-10-15 06:47:13,721] INFO [KafkaRaftServer nodeId=1] Kafka Server started (kafka.server.KafkaRaftServer)


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'logs', '--tail', '5', 'kafka-lab'], returncode=0, stdout='[2026-10-15 06:47:13,718] INFO [BrokerServer id=1] Transition from STARTING to STARTED (kafka.server.BrokerServer)\n[2026-10-15 06:47:13,719] INFO Kafka version: 4.3.1 (org.apache.kafka.common.utils.AppInfoParser)\n[2026-10-15 06:47:13,719] INFO Kafka commitId: 26b251a451ce941d (org.apache.kafka.common.utils.AppInfoParser)\n[2026-10-15 06:47:13,719] INFO Kafka startTimeMs: 1792046833719 (org.apache.kafka.common.utils.AppInfoParser)\n[2026-10-15 06:47:13,721] INFO [KafkaRaftServer nodeId=1] Kafka Server started (kafka.server.KafkaRaftServer)\n', stderr='')

You want to see a line like `Kafka Server started`. If the container keeps restarting or exits, run `docker logs kafka-lab` (no `--tail`) and read the first `ERROR`.

Which Kafka version did `latest` give you?

In [10]:
run_on_host(['docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-topics.sh', '--version'])

4.3.1


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-topics.sh', '--version'], returncode=0, stdout='4.3.1\n', stderr='')

## Step 6 — Prove it is alive, from Python

`AdminClient` is the part of the library used to *ask questions about the cluster* (it does not send messages).
`bootstrap.servers` is the first address to knock on (the two-step conversation from Step 4).

In [11]:
from confluent_kafka.admin import AdminClient

admin = AdminClient({"bootstrap.servers": "localhost:9092"})

# timeout=10: give up after 10 seconds instead of waiting forever.
cluster = admin.list_topics(timeout=10)

print("Cluster id :", cluster.cluster_id)
print("Brokers    :", [f"id={b.id} at {b.host}:{b.port}" for b in cluster.brokers.values()])
print("Topics     :", list(cluster.topics))

Cluster id : 5L6g3nShT-eMCtK--X86sw
Brokers    : ['id=1 at localhost:9092']
Topics     : []


**Expected:** one broker `id=1 at localhost:9092`, and a topics list that is empty or contains only internal names starting with `__`.

If this prints that, **you are done with setup.** 🎉

| Symptom | Likely cause | Fix |
|---|---|---|
| Hangs, then `KafkaException ... timed out` | Container not running | `docker ps`; start it (Step 5) |
| `Connection refused` | Port not published | Check `ports:` line and that nothing else uses 9092 (`ss -ltnp \| grep 9092`) |
| Connects, then errors mentioning another hostname | Wrong `ADVERTISED_LISTENERS` | Must be `localhost:9092` for a notebook on the laptop |
| Container exits immediately | Bad setting | `docker logs kafka-lab`, read the first ERROR |

## Step 7 — Stop, restart, wipe

| You want to... | Command (run in a terminal) | Messages kept? |
|---|---|---|
| Pause Kafka | `docker compose -f lab/docker-compose.yml stop` | Yes |
| Resume | `docker compose -f lab/docker-compose.yml start` | Yes |
| Remove the container | `docker compose -f lab/docker-compose.yml down` | Yes (volume stays) |
| **Wipe everything** and start fresh | `docker compose -f lab/docker-compose.yml down -v` | **No** - `-v` deletes the volume |

Leave Kafka running for the next notebook.

## Checkpoint — answer without scrolling up

1. In one sentence: what is a *broker*?
2. Why do we need the `ports` line?
3. Why must `ADVERTISED_LISTENERS` be `localhost:9092` here?
4. Why are the replication factors set to `1`?
5. What is the difference between `down` and `down -v`?

**Next: Notebook 02 — Kafka anatomy.** Topics, partitions, offsets: we create one and look at the files on disk.

---
# Appendix A — From laptop to production (reference, not a step)

Nothing to run here. Come back to this page when you wonder, "what would this look like for real?" Notebooks 02-05 explain the ideas it mentions (partitions, replication, `acks`, consumer groups).

## A.1 The shape of the system

```text
LAPTOP (this notebook)              PRODUCTION (typical minimum)
+------------------+                +----------+ +----------+ +----------+
| 1 node           |                | broker 1 | | broker 2 | | broker 3 |   3+ servers,
| broker+controller|                |(+control)| |(+control)| |(+control)|   different racks/zones
+------------------+                +----------+ +----------+ +----------+
                                    every partition has copies on 3 servers
```

## A.2 What changes from our compose file

| Our laptop file | Production | Why |
|---|---|---|
| 1 node | **3 or more nodes** | Losing one server must not lose data or availability |
| `*_REPLICATION_FACTOR: 1` | **3** | 3 copies of every partition |
| `MIN_ISR: 1` | **2** | A write counts only when at least 2 copies have it (*ISR* = in-sync replicas) |
| `image: apache/kafka:latest` | **Pinned version**, e.g. `4.2.0` | You upgrade on purpose, after testing - never by surprise |
| `PLAINTEXT` | **TLS** (encryption) + **SASL** or mTLS (login) | Anyone on the network could otherwise read or write |
| Advertised `localhost:9092` | Real DNS name per broker; often two listeners (one for apps inside the network, one for outside) | Clients must be able to reach each broker by the address it advertises |
| Default JVM memory | Explicit heap, typically 4-8 GB; leave the rest of the RAM for the OS file cache | Kafka relies on the OS cache for speed |
| Docker volume | Dedicated fast disks; `log.dirs` points at them | Disk speed and space are Kafka's main limits |
| No monitoring | Metrics + alerts: under-replicated partitions, consumer lag, disk usage | You must know about problems before users do |
| Topics appear on first use | `auto.create.topics.enable=false` | Topics are created deliberately, with chosen partitions and retention |
| Controller and broker in one process | Small cluster: combined (as now). Large cluster: **3 dedicated controllers** | Keeps cluster bookkeeping stable under load |

Also know: `unclean.leader.election.enable=false` (the default) stops an out-of-date copy from becoming leader, which would silently lose messages.

## A.3 Which configs to remember

### Tier 1 - know by heart (you will touch these constantly)

| Config | Where | Why it matters |
|---|---|---|
| `bootstrap.servers` | client | First address a client knocks on |
| `listeners` / `advertised.listeners` | broker | The cause of most "cannot connect" problems |
| `replication.factor` + `min.insync.replicas` | topic/broker | Together they decide how many broker failures you survive |
| `acks` (use `all`) | producer | How many copies must confirm before "sent" is true |
| `enable.idempotence` | producer | No duplicates when the producer retries |
| `linger.ms`, `batch.size`, `compression.type` | producer | Throughput versus latency |
| Partition count | topic | Sets the maximum parallelism; hard to reduce later |
| `retention.ms` / `retention.bytes` | topic | How long data is kept |
| `group.id` | consumer | Decides which consumers share the work |
| `auto.offset.reset` | consumer | Where a brand-new consumer starts reading |
| `enable.auto.commit` | consumer | Is progress saved automatically, or by your code? |
| `max.poll.interval.ms`, `session.timeout.ms` | consumer | Why consumers get kicked out and the group rebalances |

### Tier 2 - know they exist and what they do; look up details when needed

`retries`, `delivery.timeout.ms`, `max.in.flight.requests.per.connection`, `message.max.bytes`, `fetch.min.bytes`, `fetch.max.wait.ms`, `max.poll.records`, `heartbeat.interval.ms`, `isolation.level`, `cleanup.policy` (delete vs. compact), `segment.bytes`, `unclean.leader.election.enable`.

### Tier 3 - do not memorize; look up only when tuning a measured problem

`num.network.threads`, `num.io.threads`, socket buffer sizes, `log.flush.*`, `replica.fetch.*`, quotas, `log.cleaner.*`, and most other broker settings. Kafka's defaults are good; tuning these without measurements usually makes things worse.

### Rule of thumb

> If a setting **protects data**, **controls duplicates**, or **decides who reads what**, it is Tier 1.
> If it only **squeezes performance**, it is Tier 2 or 3.

## A.4 Where each Tier 1 item is taught

| Notebook | Topic |
|---|---|
| 02 | Partitions, replication factor, `min.insync.replicas`, retention |
| 03 | `bootstrap.servers`, producer basics, keys |
| 04 | `group.id`, `auto.offset.reset`, commits, rebalancing |
| 05 | `acks`, idempotence, `linger.ms`, `batch.size`, compression |